# Module 11 · 3：影像下游 — ViT 微調 / CLIP Zero-shot

> 影像資料（M09 圖像小節）整理好後的兩條主路線：
> **(A) 微調 ViT 分類器**（要客製、追求準確率）、**(B) CLIP zero-shot**（不訓練、快速上線）。

## 學習目標
- 用 `Trainer` 微調 **ViT** 影像分類器的最小設定（資料 `(N,C,H,W)` + 標籤）。
- 複習 **CLIP zero-shot** 作為「免訓練」選項，並比較兩者取捨。

In [ ]:
try:
    import torch
    HAS = True
except Exception:
    HAS = False
    print("需 `uv sync --extra multimodal --extra train`。說明仍可閱讀。")

## A. 微調 ViT 影像分類器

資料格式：`pixel_values (N,C,H,W)` + 標籤 `(N,)`。
用 `AutoModelForImageClassification` + `Trainer`。下方為最小設定示意（首次執行下載 ViT）。

In [ ]:
if HAS:
    try:
        import numpy as np
        from PIL import Image
        from transformers import (AutoImageProcessor, AutoModelForImageClassification,
                                   TrainingArguments, Trainer)
        from datasets import Dataset, Image as HFImage

        # 迷你資料集：兩類各 3 張合成圖
        rs = np.random.RandomState(0)
        paths, labels = [], []
        for i in range(6):
            p = f"/tmp/_vit_{i}.png"
            arr = (rs.rand(64, 64, 3) * (80 if i < 3 else 200)).clip(0, 255).astype(np.uint8)
            Image.fromarray(arr).save(p)
            paths.append(p); labels.append(0 if i < 3 else 1)
        ds = Dataset.from_dict({"image": paths, "label": labels}).cast_column("image", HFImage())

        proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
        def transform(batch):
            imgs = [im.convert("RGB") for im in batch["image"]]
            batch["pixel_values"] = proc(imgs, return_tensors="pt")["pixel_values"]
            return batch
        ds = ds.with_transform(transform)

        model = AutoModelForImageClassification.from_pretrained(
            "google/vit-base-patch16-224", num_labels=2, ignore_mismatched_sizes=True)
        args = TrainingArguments(output_dir="/tmp/vit_demo", num_train_epochs=1,
                                 per_device_train_batch_size=2, report_to=[], no_cuda=True)

        def collate(examples):
            import torch
            return {"pixel_values": torch.stack([e["pixel_values"] for e in examples]),
                    "labels": torch.tensor([e["label"] for e in examples])}
        trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collate)
        trainer.train()
        print("✓ ViT 分類器最小微調完成（demo）。")
    except Exception as e:
        print("（未能下載 ViT 或環境不足，略過實際訓練）", e)

## B. CLIP Zero-shot（免訓練）

不想訓練？CLIP 用文字 prompt 直接分類，改類別只要改文字（見 M09 圖像 03/05）。

In [ ]:
if HAS:
    try:
        import numpy as np
        from PIL import Image
        from transformers import CLIPModel, CLIPProcessor
        clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        cproc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        img = Image.fromarray((np.random.RandomState(1).rand(224, 224, 3)*255).astype(np.uint8))
        labels = ["a cat", "a dog", "a flower"]
        inp = cproc(text=[f"a photo of {l}" for l in labels], images=img,
                    return_tensors="pt", padding=True)
        with torch.no_grad():
            probs = clip(**inp).logits_per_image.softmax(dim=1)[0]
        print("CLIP zero-shot:", {l: round(p.item(), 3) for l, p in zip(labels, probs)})
    except Exception as e:
        print("（未能下載 CLIP，略過）", e)

## 小結

| 路線 | 是否訓練 | 何時用 |
|:--|:--|:--|
| A 微調 ViT | 是 | 有標註、追求準確率、固定類別 |
| B CLIP zero-shot | 否 | 快速上線、類別常變、缺標註 |

實務常**先用 CLIP zero-shot 起步**，累積標註後再微調 ViT 提升表現。